# Financial Document Analysis Agent

**CSC3173 Artificial Intelligence: Programming Assignment 02**

This notebook builds an agent that answers financial-analysis questions about **Apple Inc.**
using its annual reports (Form 10-K) for fiscal years **2023, 2024 and 2025**.

The design follows the Medical Document (WHO statistics) Agent from class: PDFs are split into
chunks, embedded and indexed with FAISS, and a LangGraph loop lets a planner choose between
`RETRIEVE`, `CALCULATE`, `REPORT` and `FINISH`. The health-specific parts were replaced with
financial ones: a table-aware retrieval tool for the 10-K net sales tables, a Python calculation
tool, and a report generator that writes a PDF with charts and page citations.

Notebook outline:

| Stage | Section |
|---|---|
| 0 | Agentic components (Medical agent vs Financial agent) |
| A | Document processing |
| B | Semantic retrieval |
| C | Financial retrieval tool |
| D | Financial calculation tool |
| E | Report generation tool |
| F | LangGraph agent (state, planner, nodes, routing) |
| G | Test questions Q1 to Q3 and PDF report |

## Setup
Mount Google Drive (on Colab) and set the project paths. The three 10-K PDFs go in `Finance_AI_Agent/data/` (or directly in `Finance_AI_Agent/`).

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Demo/AI/Finance_AI_Agent"
else:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = PROJECT_DIR
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Data dir   :", DATA_DIR)
print("Output dir :", OUTPUT_DIR)

### Package installation

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate
!pip install -q langgraph
!pip install -q fpdf2

In [ ]:
import fitz  # PyMuPDF
import re, glob, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

## A. Document Processing

Each PDF is opened with PyMuPDF and the text of every page is kept together with its metadata
(source file, page number, fiscal year).

The class notebook took the year from the file name. Here the fiscal year is read from the cover
page of the filing (*"For the fiscal year ended September 27, 2025"*), so renamed files still get
the correct year. Only PDFs whose first page says *FORM 10-K* are loaded, which keeps the
generated report PDF out of the index if it ends up in the same folder.

In [ ]:
def is_10k_filing(doc) -> bool:
    return "FORM 10-K" in doc[0].get_text().upper()

def detect_fiscal_year(doc, path: str) -> str:
    cover = doc[0].get_text().replace("\xa0", " ")
    match = re.search(r"fiscal year ended\s+[A-Za-z]+\s+\d{1,2},\s+(20\d{2})", cover, re.IGNORECASE)
    if match:
        return match.group(1)
    match = re.search(r"(20\d{2})", os.path.basename(path))
    return match.group(1) if match else "unknown"

def clean_page_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)
    return text.strip()

def extract_pdf_pages(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    if not is_10k_filing(doc):
        return []
    fiscal_year = detect_fiscal_year(doc, pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        text = clean_page_text(page.get_text())
        if text:
            pages.append({"text": text, "page": i, "year": fiscal_year,
                          "source": os.path.basename(pdf_path)})
    return pages

pdf_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.pdf")))
assert pdf_paths, f"No PDF files found in {DATA_DIR}"

all_pages = []
for path in pdf_paths:
    pages = extract_pdf_pages(path)
    if pages:
        all_pages.extend(pages)
        print(f"{os.path.basename(path):28s} -> FY{pages[0]['year']}, {len(pages)} pages")

AVAILABLE_YEARS = sorted({p["year"] for p in all_pages if p["year"] != "unknown"})
print("Fiscal years available:", AVAILABLE_YEARS)

In [ ]:
# quick lookup of the full text of any page, used later by the table parser
page_lookup = {(p["source"], p["page"]): p for p in all_pages}

summary = (pd.DataFrame(all_pages)
           .assign(words=lambda d: d["text"].str.split().str.len())
           .groupby(["year", "source"])
           .agg(pages=("page", "count"), words=("words", "sum")))
summary

### Chunking with source / year / page metadata

Pages are split into overlapping word windows. Every chunk keeps the metadata of its page so
answers can always be traced back to a page in a filing.

In [ ]:
def chunk_text(text: str, chunk_size: int = 250, overlap: int = 50) -> list[str]:
    words = text.split()
    if len(words) <= chunk_size:
        return [" ".join(words)]
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
        start += chunk_size - overlap
    return chunks

def build_chunks(pages: list[dict]) -> list[dict]:
    out = []
    for page in pages:
        for piece in chunk_text(page["text"]):
            out.append({"text": piece, "page": page["page"], "year": page["year"],
                        "source": page["source"]})
    return out

chunks = build_chunks(all_pages)
print(f"Total chunks: {len(chunks)}")
print(pd.Series([c["year"] for c in chunks]).value_counts().sort_index().to_string())